# Organizando a saída por Copa

## Task 3: organizar a saída por Copa e carregar uma Copa nova sem apagar o histórico

O painel pergunta por **uma Copa** de cada vez, e a Copa de 2026 precisa entrar sem apagar as outras: a saída fica
organizada **por Copa**.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-particionamento")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
RAW_2026 = "../../data/raw_2026"
EVENTOS = "../../data/events"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

## Duas coisas com o mesmo nome

| | Partição **em memória** | Partição **em disco** |
|---|---|---|
| O que é | um pedaço do dado, processado por uma task | uma pasta com os arquivos de um valor de coluna |
| Quem controla | `repartition`, `coalesce`, `shuffle.partitions` | `partitionBy`, na escrita |

## partitionBy: a coluna vira pasta

Cada ano virou uma pasta `ano=2018`. Na leitura, o nome da pasta volta a ser coluna:

## Partition pruning: não abrir o que não precisa

**`PartitionFilters`** com `ano = 2018`: o Spark nem abre as outras pastas (**partition pruning**).

Fora da partição, o filtro vai para **`PushedFilters`**: o Spark abre todas as pastas.

**Como escolher a coluna:** a que mais aparece nos filtros, com **poucos valores diferentes** (ano, país; não um
identificador).

## O problema dos arquivos pequenos

**Cada task grava um arquivo em cada pasta que recebe dados.**

Cada task gravou um arquivo em cada pasta. Simulando 20 tasks:

**Cada arquivo tem um custo fixo** para abrir e ler. Solução: `repartition("ano")` antes de gravar, um arquivo por
pasta.

## Uma Copa nova na saída por Copa

E o `overwrite`?

Só sobrou 2026: o overwrite **estático** (padrão) apagou a pasta inteira. Com `partitionOverwriteMode` em `dynamic`,
ele substitui **só as pastas do lote novo**:

19 Copas, e o total não muda se o job rodar de novo: **idempotência**.

| Modo | Rodou duas vezes | Histórico |
|---|---|---|
| `append` | duplicou | preservado |
| `overwrite` (estático) | ok | **apagado** |
| `overwrite` com `dynamic` | ok | **preservado** |

## Resultado da task 3

Saída por Copa, um arquivo por pasta, e uma Copa nova sem apagar nem duplicar. Próxima task: o pipeline fora do
notebook.

## Para consultar depois

- **Mais de uma coluna:** `partitionBy("ano", "fase")` cria pastas dentro de pastas (`ano=2018/fase=Final`). Cuidado:
  o número de pastas é a combinação dos valores, e cada pasta pode ficar com pouco dado.
- **`maxRecordsPerFile`:** `.option("maxRecordsPerFile", 50000)` limita as linhas por arquivo e quebra só as pastas
  grandes.
- **Sem transação:** se o job falhar no meio de um overwrite, parte do dado pode ter sido apagada. Formatos como o
  Delta Lake resolvem isso (o notebook extra do vídeo 5.2).

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()